In [10]:
import requests
import pandas as pd

def fetch_nairobi_retail_nodes():
    """ Querying OpenStreetMap for Nairobi retail nodes..."""
    
    overpass_url= "http://overpass-api.de/api/interpreter"
    overpass_query="""
    [out:json][timeout:25];
    (
        node["amenity"="bar"](-1.44,36.65,-1.15,37.10);
        node["amenity"="pub"](-1.44,36.65,-1.15,37.10);
        node["shop"="supermarket"](-1.44,36.65,-1.15,37.10);
        node["shop"="convenience"](-1.44,36.65,-1.15,37.10);
    );
    out center;    
    """

    headers={
        "User-Agent": "Adiel_Nairobi_RTM_Research_Project/1.0"
    }

    response=requests.post(overpass_url,data={'data': overpass_query}, headers=headers)

    if response.status_code == 200:
        data =  response.json()
        nodes=[]
        for element in data["elements"]:

            tags=element.get("tags",{})
            name= tags.get('name', 'Unnamed Store')

            if tags.get("amenity") in ["bar", "pub"]:
                node_type= 'Bar/Pub'
            elif tags.get("shop")=="supermarket":
                node_type="Supermarket"
            elif tags.get("shop")=="convenience":
                node_type= "Convenience/Duka"
            else:
                node_type="Other"
            
            nodes.append({
                "Node_ID": element["id"],
                'Name':name,
                "Type": node_type,
                "Latitude": element["lat"],
                "Longitude": element["lon"]
            })

        df_nodes = pd.DataFrame(nodes)
        print(f"Successfully extracted {len(df_nodes)} delivery nodes in Nairobi!")
        return df_nodes
    else:
        print(f"Error: Failed to fetch data. Status code: {response.status_code}")
        return None
if __name__ == "__main__":
    nairobi_df=fetch_nairobi_retail_nodes()

    if nairobi_df is not None:
        print(nairobi_df["Type"].value_counts())
        print("\nFirst 5 Rows:")
        print(nairobi_df.head())

        nairobi_df.to_csv("Nairobi_delivery_nodes.csv", index=False)
        print("\n Saved as Nairobi CSV")
        

Successfully extracted 1026 delivery nodes in Nairobi!
Type
Bar/Pub             631
Supermarket         284
Convenience/Duka    111
Name: count, dtype: int64

First 5 Rows:
    Node_ID           Name         Type  Latitude  Longitude
0  30092040      Carrefour  Supermarket -1.298407  36.763324
1  30210335         Barizi      Bar/Pub -1.292564  36.786825
2  30324274     Soho's Bar      Bar/Pub -1.262588  36.805176
3  30324343      Carrefour  Supermarket -1.260903  36.801831
4  30402750  K1 Klub House      Bar/Pub -1.268460  36.811872

 Saved as Nairobi CSV
